# 2. Router/Worker, 두 단계 필터와 구독

근거: [설계 글 §2·§3.1–3.2](https://x.com/DeepSeekHarness/status/2116857257245987236?s=20).

**실행:** Python (Harness Lab) 커널에서 위에서 아래로 실행하세요. 각 노트북은 새 커널에서 독립 실행됩니다. 기본 셀은 외부 API를 호출하지 않습니다.

## 구조로 역할을 제한하기
Router에는 작업 실행 도구를 주지 않고 배정과 외부 응답만 맡깁니다. Worker는 자기 작업 문맥만 받습니다. 모든 후보 사건을 항상 중앙 모델에 보내면 Router가 부담을 떠안습니다. 이미 구독된 Thread의 후속 사건은 코드로 전달합니다.

아래 semantic 함수는 LLM 판단을 흉내 내는 결정적 대역입니다. 운영에서는 structured output 모델로 교체할 수 있습니다. 코드 필터는 허용된 발신원·중복 여부 같은 확실한 조건을 처리하고, 의미 판단은 후보에만 수행합니다. Router의 가시성(observed)과 모델의 재배정 호출(decisions)은 다른 지표입니다.

In [1]:
from dataclasses import dataclass
from collections import defaultdict
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

@dataclass(frozen=True)
class Trigger:
    id: str
    thread: str
    channel: str
    text: str
    trusted: bool=True

class WorkerState(TypedDict):
    task: str
    conclusion: str
wb=StateGraph(WorkerState)
wb.add_node('work',lambda s:{'conclusion':'로컬 검토: '+s['task']})
wb.add_edge(START,'work'); wb.add_edge('work',END)
worker=wb.compile()

class Router:
    def __init__(self):
        self.seen=set(); self.subscribers=defaultdict(set)
        self.inboxes=defaultdict(list); self.observed=[]
        self.decisions=0; self.semantic_calls=0
    def semantic(self,event):
        self.semantic_calls+=1
        return any(word in event.text for word in ['버그','수정','취소'])
    def admit(self,event):
        if not event.trusted or event.id in self.seen: return []
        self.seen.add(event.id)
        if not self.semantic(event): return []
        self.observed.append(event.id)
        targets=sorted(self.subscribers[event.thread])
        if not targets:
            self.decisions+=1
            targets=[f'worker-{self.decisions}']
            self.subscribers[event.thread].update(targets)
        for target in targets: self.inboxes[target].append(event)
        return targets

r=Router()
a=Trigger('e1','im:topic-42','im','버그 재현')
b=Trigger('e2','im:topic-42','im','수정 방향 변경')
assert r.admit(a)==['worker-1']
assert r.admit(b)==['worker-1']
assert r.decisions==1 and r.observed==['e1','e2']
assert r.admit(a)==[]  # 중복 전달 방지
assert r.admit(Trigger('noise','spam','im','버그',False))==[]
assert r.semantic_calls==2
# 같은 Worker가 GitHub PR에도 구독: 채널을 바꿔도 작업 지속
r.subscribers['github:pr-42'].add('worker-1')
r.admit(Trigger('e3','github:pr-42','github','수정 검토'))
# Thread에 두 Worker가 구독: peer에게 개별 연락하지 않고 같은 사건 전달
r.subscribers['github:pr-42'].add('reviewer')
assert r.admit(Trigger('e4','github:pr-42','github','버그 검토'))==['reviewer','worker-1']
assert r.decisions==1
result=worker.invoke({'task':r.inboxes['worker-1'][0].text,'conclusion':''})
print('Router 재배정:',r.decisions, '관찰:',r.observed)
print('Worker 결과:',result['conclusion'])

Router 재배정: 1 관찰: ['e1', 'e2', 'e3', 'e4']
Worker 결과: 로컬 검토: 버그 재현


## 이 예제가 생략한 것
Worker 실행은 여기서는 invoke로 분리해 보여줍니다. 실제 병행 실행은 04·07에서 합니다. 발신자 신뢰 여부는 데모 필드이며 인증 구현이 아닙니다. 구독은 쓰기 잠금이 아니므로 두 Worker가 같은 파일을 수정하는 충돌을 방지하지 않습니다. 잡담 필터가 후속 작업 취소를 놓치지 않도록 정책 테스트가 필요합니다.

원문 Router batching은 추론 중 쌓인 사건을 다음 추론에 묶어 넣습니다. 무제한 대기가 아니라 최대 개수·대기시간·문맥 예산을 정해야 합니다.

In [2]:
def batches(events, limit):
    if limit<1: raise ValueError('양수 limit 필요')
    return [events[i:i+limit] for i in range(0,len(events),limit)]
queue=[f'e{i}' for i in range(7)]
groups=batches(queue,3)
assert list(map(len,groups))==[3,3,1]
assert sum(groups,[])==queue
print('7개 사건, 최대 3개씩 처리:', groups)

7개 사건, 최대 3개씩 처리: [['e0', 'e1', 'e2'], ['e3', 'e4', 'e5'], ['e6']]


## 직접 바꿔 보기

이미 구독된 Thread에서 완전히 다른 작업이 시작되면 어떻게 재배정할까요? 위 Router에서 worker-1을 해제하고 새 사건을 넣어 decisions 증가를 확인하세요.

## 해설과 참고 답안

r.subscribers["im:topic-42"].discard("worker-1") 후 새 ID의 사건을 admit하면 새 Worker가 배정됩니다. 운영에서는 Worker의 hand-back 사건과 해제 작업을 원자적으로 처리해 유실·재전달 루프를 막아야 합니다.

## 공개 프로토타입과의 차이

이 노트북의 semantic → routing 순서는 교육용 단순화입니다. 공개 프로토타입 Intake는 SourceScope 필터 뒤 **구독 fan-out을 먼저** 하고, Ticket Worker가 소유한 사람 후속 메시지는 Eval을 생략합니다. bot/CI 및 context-only follow는 다르게 처리합니다. 또한 실제 Main Session에는 bash/read 도구가 있고 실행 역할 제한은 프롬프트 규칙입니다. 09–10에서 차이를 실습합니다.